In [1]:
import os

# 在导入任何网络库之前清除代理设置
for var in [
    "HTTP_PROXY",
    "HTTPS_PROXY",
    "http_proxy",
    "https_proxy",
    "ALL_PROXY",
    "all_proxy",
]:
    os.environ.pop(var, None)

# 强制设置不代理本地地址
os.environ["NO_PROXY"] = "localhost,127.0.0.1"


In [2]:
from langchain_ollama import ChatOllama
import httpx
BASE_URL = "http://127.0.0.1:11434"
LLM_MODEL = "gpt-oss:20b-cloud"

llm = ChatOllama(
    model=LLM_MODEL,
    base_url=BASE_URL,
    http_client=httpx.Client(trust_env=False),
    http_async_client=httpx.AsyncClient(trust_env=False),
    reasoning=False
)
llm.invoke("Hi there!")

AIMessage(content='Hello! 👋 How can I help you today?', additional_kwargs={}, response_metadata={'model': 'gpt-oss:20b-cloud', 'created_at': '2026-03-12T05:56:32.325847429Z', 'done': True, 'done_reason': 'stop', 'total_duration': 617377245, 'load_duration': None, 'prompt_eval_count': 65, 'prompt_eval_duration': None, 'eval_count': 82, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:20b-cloud', 'model_provider': 'ollama'}, id='lc_run--019ce09e-32cb-7cb2-b19d-845f8c16cfa0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 65, 'output_tokens': 82, 'total_tokens': 147})

In [17]:
from utils.schemas import ChunkMetadata
# 调用llm从用户的问题中提取关键词，返回关键词字典
def extract_filters(user_query: str) -> dict:

    llm_structured = llm.with_structured_output(ChunkMetadata)

    prompt = f"""Extract metadata filters from the query. Output valid JSON only. Use null (not None) for missing fields.

                USER QUERY: {user_query}

                COMPANY MAPPINGS:
                - Amazon/AMZN -> amazon
                - Google/Alphabet/GOOGL/GOOG -> google
                - Apple/AAPL -> apple
                - Microsoft/MSFT -> microsoft
                - Tesla/TSLA -> tesla
                - Nvidia/NVDA -> nvidia
                - Meta/Facebook/FB -> meta

                DOC TYPE:
                - Annual report -> 10-k
                - Quarterly report -> 10-q
                - Current report -> 8-k

                EXAMPLES:
                "Amazon Q3 2024 revenue" -> {{"company_name": "amazon", "doc_type": "10-q", "fiscal_year": 2024, "fiscal_quarter": "q3"}}
                "Apple 2023 annual report" -> {{"company_name": "apple", "doc_type": "10-k", "fiscal_year": 2023}}
                "Tesla profitability" -> {{"company_name": "tesla"}}

                Extract metadata:
                """

    metadata = llm_structured.invoke(prompt)
    filters = metadata.model_dump(exclude_none=True)

    return filters
extract_filters("what is amazon's revenue in 2023?")

{'company_name': 'amazon', 'doc_type': '10-k', 'fiscal_year': 2023}

In [7]:
from utils.schemas import RankingKeywords
# 调用llm生成与问题可能相关的keyword，便于之后提取内容进行排序
def generate_ranking_keywords(user_query: str) -> list:
    # ALT + Z
    prompt = f"""Generate EXACTLY 5 financial keywords from SEC filings terminology.

                USER QUERY: {user_query}

                USE EXACT TERMS FROM 10-K/10-Q FILINGS:

                STATEMENT HEADINGS:
                "consolidated statements of operations", "consolidated balance sheets", "consolidated statements of cash flows", "consolidated statements of stockholders equity"

                INCOME STATEMENT:
                "revenue", "net revenue", "cost of revenue", "gross profit", "operating income", "net income", "earnings per share"

                BALANCE SHEET:
                "total assets", "cash and cash equivalents", "total liabilities", "stockholders equity", "working capital", "long-term debt"

                CASH FLOWS:
                "cash flows from operating activities", "net cash provided by operating activities", "cash flows from investing activities", "free cash flow", "capital expenditures"

                RULES:
                - Return EXACTLY 5 keywords
                - Use exact phrases from SEC filings
                - Match query topic (revenue -> revenue terms, cash -> cash flow terms)
                - Use "cash flows" (plural), "stockholders equity"

                EXAMPLES:
                "revenue analysis" -> ["revenue", "net revenue", "total revenue", "consolidated statements of operations", "net sales"]
                "cash flow performance" -> ["consolidated statements of cash flows", "cash flows from operating activities", "net cash provided by operating activities", "free cash flow", "operating activities"]
                "balance sheet strength" -> ["consolidated balance sheets", "total assets", "stockholders equity", "cash and cash equivalents", "long-term debt"]

                Generate EXACTLY 5 keywords as JSON object:
                {{"keywords": ["keyword1", "keyword2", "keyword3", "keyword4", "keyword5"]}}
                """

    llm_structured = llm.with_structured_output(RankingKeywords)
    result = llm_structured.invoke(prompt)

    return result.keywords
generate_ranking_keywords("what is amazon's revenue in 2023?")

['revenue',
 'net revenue',
 'gross profit',
 'consolidated statements of operations',
 'operating income']